# Data Preparation Repository

This notebook demonstrates how data is prepared, including downloading and ground truth computation.

In [5]:
import gzip
import json
import numpy as np
import h5py
from tqdm import tqdm
import glob
import hnswlib
from sklearn.datasets import make_blobs

In [ ]:
# Create directory if it doesn't exist
! makdir -p ~/experiments/data

In [ ]:
# Dataset: DeepImage
# Download: http://ann-benchmarks.com/deep-image-96-angular.hdf5
# Save the data to ../experiments/data/deep-image-96-angular.hdf5

! wget http://ann-benchmarks.com/deep-image-96-angular.hdf5 -P ~/experiments/data/

In [ ]:
# Dataset: GloVe-100
# Download: http://ann-benchmarks.com/glove-100-angular.hdf5
# Save the data to ../experiments/data/glove-100-angular.hdf5

! wget http://ann-benchmarks.com/glove-100-angular.hdf5 -P ~/experiments/data/

In [6]:
# Dataset: MS MARCO V1
! wget https://rgw.cs.uwaterloo.ca/pyserini/data/msmarco-passage-openai-ada2.tar -P collections/
! tar xvf collections/msmarco-passage-openai-ada2.tar -C collections/
print("Downloaded and extracted MS MARCO V1 dataset")
msmarco_v1_data_vecs = []
for i in tqdm(range(0, 89)):
    with gzip.open(str(i) + '.jsonl.gz', 'rt', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            msmarco_v1_data_vecs.append(data['vector'])
msmarco_v1_data_vecs = np.array(msmarco_v1_data_vecs, dtype=np.float32)
print(msmarco_v1_data_vecs.shape)

msmarco_v2_query_vecs = []
with gzip.open('topics.msmarco-passage.dev-subset.openai-ada2.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        msmarco_v2_query_vecs.append(data['vector'])
msmarco_v2_query_vecs = np.array(msmarco_v2_query_vecs, dtype=np.float32)
print(msmarco_v2_query_vecs.shape)

# using hnswlib to compute ground truth
bf_msmarco_v1 = hnswlib.BFIndex(space='cosine', dim=msmarco_v1_data_vecs.shape[1])
bf_msmarco_v1.init_index(max_elements=msmarco_v1_data_vecs.shape[0])
bf_msmarco_v1.add_items(msmarco_v1_data_vecs)
gt_msmarco_v1, _ = bf_msmarco_v1.knn_query(msmarco_v2_query_vecs, k=1000)

with h5py.File("msmarco.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=msmarco_v2_query_vecs)
    h5f.create_dataset("train", data=msmarco_v1_data_vecs)
    h5f.create_dataset("neighbors", data=gt_msmarco_v1)

print("Saved MS MARCO V1 dataset and ground truth to msmarco.hdf5")
! mv msmarco.hdf5 ~/experiments/data/


--2026-06-11 16:51:56--  https://rgw.cs.uwaterloo.ca/pyserini/data/msmarco-passage-openai-ada2.tar
Resolving rgw.cs.uwaterloo.ca (rgw.cs.uwaterloo.ca)... 129.97.167.168
Connecting to rgw.cs.uwaterloo.ca (rgw.cs.uwaterloo.ca)|129.97.167.168|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 116175247360 (108G) [application/x-tar]
Saving to: ‘collections/msmarco-passage-openai-ada2.tar’

penai-ada2.tar        4%[                    ]   4.86G  5.93MB/s    eta 6h 35m ^C
msmarco-passage-openai-ada2/
msmarco-passage-openai-ada2/0.jsonl.gz
msmarco-passage-openai-ada2/10.jsonl.gz
msmarco-passage-openai-ada2/11.jsonl.gz
msmarco-passage-openai-ada2/12.jsonl.gz
tar: msmarco-passage-openai-ada2: Cannot mkdir: No such file or directory
tar: Unexpected EOF in archive
tar: msmarco-passage-openai-ada2: Cannot utime: No such file or directory
tar: Error is not recoverable: exiting now
Downloaded and extracted MS MARCO V1 dataset


  0%|                                                                                            | 0/89 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '0.jsonl.gz'

In [ ]:
# Dataset: MS MARCO V2

# Data vectors
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_00.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_01.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_02.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_03.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_04.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_05.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_06.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_07.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_08.npy
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/passages_npy/msmarco_v2.1_doc_segmented_09.npy

# Query vectors
! wget https://huggingface.co/datasets/Cohere/msmarco-v2.1-embed-english-v3/resolve/main/queries_jsonl/queries.jsonl.gz

ms_marco_v2_queries = []
with gzip.open('query/queries.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        ms_marco_v2_queries.append(json.loads(line))
ms_marco_v2_query_vecs = [query['emb'] for query in ms_marco_v2_queries]
ms_marco_v2_query_vecs = np.array(ms_marco_v2_query_vecs).astype(np.float32)

ms_marco_v2_data = []
for e_path in sorted(glob.glob("data/*.npy")):
  ms_marco_v2_data.append(np.load(e_path))
ms_marco_v2_data_vecs = np.vstack(ms_marco_v2_data).astype(np.float32)

# using hnswlib to compute ground truth
bf_msmarco_v1 = hnswlib.BFIndex(space='cosine', dim=ms_marco_v2_data_vecs.shape[1])
bf_msmarco_v1.init_index(max_elements=ms_marco_v2_data_vecs.shape[0])
bf_msmarco_v1.add_items(ms_marco_v2_data_vecs)
gt_msmarco_v2, _ = bf_msmarco_v1.knn_query(ms_marco_v2_query_vecs, k=1000)

with h5py.File("cohere.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=ms_marco_v2_query_vecs)
    h5f.create_dataset("train", data=ms_marco_v2_data_vecs)
    h5f.create_dataset("neighbors", data=gt_msmarco_v2)
    
print("Saved MS MARCO V2 dataset and ground truth to cohere.hdf5")
! mv cohere.hdf5 ~/experiments/data/

In [ ]:
# Dataset: Laion-I2I and Laion-T2I
# download image embeddings
for i in range(0, 31):
    ! wget --tries=100 https://deploy.laion.ai/8f83b608504d46bb81708ec86e912220/embeddings/img_emb/img_emb_{i}.npy
    
img_emb = []
for i in range(0, 31):
    img_emb_i = np.load(f'img/img_emb_{i}.npy')
    img_emb.append(img_emb_i)
laion_i2i_all_data = np.vstack(img_emb)
print(laion_i2i_all_data.shape)
# (30656115, 512)

# Randomly sample 10,000 rows without replacement as query vectors
sampled_indices = np.random.choice(laion_i2i_all_data.shape[0], 10000, replace=False)
laion_image_query_vectors = laion_i2i_all_data[sampled_indices]

# Get the remaining rows
remaining_indices = np.setdiff1d(np.arange(laion_i2i_all_data.shape[0]), sampled_indices)
laion_image_data_vectors = laion_i2i_all_data[remaining_indices]

# using hnswlib to compute ground truth
bf_laion_images = hnswlib.BFIndex(space='cosine', dim=laion_image_data_vectors.shape[1])
bf_laion_images.init_index(max_elements=laion_image_data_vectors.shape[0])
bf_laion_images.add_items(laion_image_data_vectors)
gt_laion_i2i, _ = bf_laion_images.knn_query(laion_image_query_vectors, k=1000)


with h5py.File("laion_image.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=laion_image_query_vectors)
    h5f.create_dataset("train", data=laion_image_data_vectors)
    h5f.create_dataset("neighbors", data=gt_laion_i2i)

print("Saved Laion-I2I dataset and ground truth to laion_image.hdf5")
! mv laion_image.hdf5 ~/experiments/data/


# download text embeddings
txt_emb = []
for i in range(0, 31):
    txt_emb_i = np.load(f'txt/text_emb_{i}.npy')
    txt_emb.append(txt_emb_i)
merged_txt_emb = np.vstack(txt_emb)
print(merged_txt_emb.shape)

with h5py.File("laion_text_ng.hdf5", "w") as h5f:
    h5f.create_dataset("train", data=merged_txt_emb)
print("Saved Laion-T2I text embeddings to laion_text_ng.hdf5")
! mv laion_text_ng.hdf5 ~/experiments/data/

# sample 10,000 query vectors
sampled_indices_txt = np.random.choice(merged_txt_emb.shape[0], 10000, replace=False)
laion_text_query_vectors = merged_txt_emb[sampled_indices_txt]
# compute ground truth for text query vectors over image data vectors
gt_laion_t2i, _ = bf_laion_images.knn_query(laion_text_query_vectors, k=1000)
with h5py.File("laion_text_query_groundtruth.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=laion_text_query_vectors)
    h5f.create_dataset("neighbors", data=gt_laion_t2i)


In [ ]:
# Dataset: Uniform Clustered Data
dim = 100
num_elements = 10000000
n_clusters   = 5000
std = 0.01
low = -0.1
high = 0.1

query_num = 10000

uniform_data_vecs, _ = make_blobs(  # type: ignore
    n_samples=num_elements,  
    n_features=dim,  
    centers=n_clusters,  
    center_box=(low, high),  
    cluster_std=std,  
    shuffle=True,
    random_state=42,
)
uniform_data_vecs = uniform_data_vecs.astype(np.float32)

sample_ids = np.random.choice(np.arange(num_elements), query_num, replace=False)

uniform_query_vecs = uniform_data_vecs[sample_ids]

# using hnswlib to compute ground truth
bf_uniform = hnswlib.BFIndex(space='cosine', dim=uniform_data_vecs.shape[1])
bf_uniform.init_index(max_elements=uniform_data_vecs.shape[0])
bf_uniform.add_items(uniform_data_vecs)
gt_uniform, _ = bf_uniform.knn_query(uniform_query_vecs, k=1000)    

with h5py.File("cluster_mg_uniform_100d.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=uniform_query_vecs)
    h5f.create_dataset("train", data=uniform_data_vecs)
    h5f.create_dataset("neighbors", data=gt_uniform)
    
print("Saved Uniform Clustered Data and ground truth to cluster_mg_uniform_100d.hdf5")
! mv cluster_mg_uniform_100d.hdf5 ~/experiments/data/

In [ ]:
# Dataset: Zipfian Clustered Data
def zipf_samples(N: int, total: int, s: float = 1.5, base: int = 50) -> np.ndarray:
    """
    Generate Zipf-distributed sample counts using multinomial sampling.

    Parameters:
        N (int): Number of ranked items.
        total (int): Total number of samples.
        s (float): Zipf exponent (higher = more skewed).
        base (int): Minimum count per item (ensures no zeros if base > 0).
        seed (int): Random seed for reproducibility.

    Returns:
        np.ndarray: Array of sample counts, shape (N,), sum = total.
    """
    rng = np.random.default_rng(42)

    if total < N * base:
        raise ValueError("Total must be at least N * base to allow non-zero base values.")

    # Step 1: Calculate Zipf probabilities
    ranks = np.arange(1, N + 1)
    weights = 1 / np.power(ranks, s)
    probs = weights / weights.sum()

    # Step 2: Sample remaining counts
    remaining_total = total - N * base
    extra_samples = rng.multinomial(remaining_total, probs)

    # Step 3: Add base to avoid zeros
    samples = extra_samples + base
    return samples

dim = 100
num_elements = 10000000
n_clusters   = 5000
std = 0.01
low = -0.1
high = 0.1
base = 100

query_num = 10000

zipf_data_vecs, _ = make_blobs( # type: ignore
    n_samples=zipf_samples(N=n_clusters, total=num_elements, s=1, base=base),
    n_features=dim,
    centers=np.random.uniform(low=low, high=high, size=(n_clusters, dim)),
    cluster_std=std,
    shuffle=True,
    random_state=42,
)
zipf_data_vecs = zipf_data_vecs.astype(np.float32)

sample_ids = np.random.choice(np.arange(num_elements), query_num, replace=False)
zipf_query_vecs = zipf_data_vecs[sample_ids]

# using hnswlib to compute ground truth
bf_zipf = hnswlib.BFIndex(space='cosine', dim=zipf_data_vecs.shape[1])
bf_zipf.init_index(max_elements=zipf_data_vecs.shape[0])
bf_zipf.add_items(zipf_data_vecs)
gt_zipf, _ = bf_zipf.knn_query(zipf_query_vecs, k=1000) 

with h5py.File("cluster_mg_zipf_100d.hdf5", "w") as h5f:
    h5f.create_dataset("test", data=zipf_query_vecs)
    h5f.create_dataset("train", data=zipf_data_vecs)
    h5f.create_dataset("neighbors", data=gt_zipf)
    
print("Saved Zipfian Clustered Data and ground truth to cluster_mg_zipf_100d.hdf5")
! mv cluster_mg_zipf_100d.hdf5 ~/experiments/data/